# Export the deployment model for the real-time app

This notebook reuses, unchanged, the data, feature, model and training cells of the hybrid quantum-classical notebook, then trains the **final model (hands + pose + lips, 4-qubit circuits)** and writes one file, `slt_deploy_<N>signs.pt`, that the real-time app loads.

* **Data split:** 19 of the 21 signers train the model. One signer is used for validation and one for a sanity-check test; neither is trained on. (The paper numbers come from the separate 3-signer-test runs.)
* **Run on Kaggle:** Add Input *Google - Isolated Sign Language Recognition*, accelerator GPU, then Run All. Set `DEPLOY_SIGNS` in the second cell first (50 is the default; 10 is faster and easier to demo).
* **Output:** download `slt_deploy_50signs.pt` from the Output tab (right side, `/kaggle/working`).
* **Time:** about as long as one of your 50-sign training runs (your earlier logs are the best estimate).

In [ ]:
import os, json, time, math, copy, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from joblib import Parallel, delayed
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

_default_base = Path("/kaggle/input/competitions/asl-signs")
if not _default_base.exists():
    _default_base = Path("/kaggle/input/asl-signs")
BASE = Path(os.environ.get("ASL_BASE", str(_default_base)))
WORK = Path(os.environ.get("ASL_WORK", "/kaggle/working"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- config ----------------
MODE      = "paper10"   # "quick": 50 signs | "full": 250 signs | "paper10": 10 signs + random split (base-paper protocol)
NUM_SIGNS = {"quick": 50, "full": 250, "paper10": 10}[MODE]
SPLIT_MODE = "random" if MODE == "paper10" else "signer"   # signer = unseen-signer test (strict)
# SPLIT_MODE = "signer"   # uncomment: strict unseen-signer number on the same 10 signs
SIGN_SEED = 42
SEEDS     = [0, 1, 2]
EPOCHS    = 50
T_CACHE   = 64
print(DEVICE, torch.__version__)

use_amp = DEVICE == "cuda"
amp = lambda: torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp)


In [ ]:
# ---- deployment settings (override the notebook defaults above) ----
DEPLOY_SIGNS = 50        # vocabulary of the live system: 10 or 50
NUM_SIGNS  = DEPLOY_SIGNS
SPLIT_MODE = "signer"    # whole-signer split, never the random split
SEEDS      = []          # the multi-seed sweeps are not needed here
EPOCHS     = 50          # same recipe as the paper
print(f"deployment vocabulary: {NUM_SIGNS} signs | device {DEVICE}")

## 1. Data

In [ ]:
# ============================================================
# ISLR DATASET LOADING + FRAME LAYOUT VALIDATION
# ============================================================
required_files = ["train.csv", "sign_to_prediction_index_map.json", "train_landmark_files"]
for name in required_files:
    path = BASE / name
    assert path.exists(), f"Missing required dataset item: {path}"

print("Dataset path:", BASE)
print("Dataset files:", os.listdir(BASE))

LH0, POSE0, RH0 = 468, 489, 522
POSE_KEEP = [0, 11, 12, 13, 14, 15, 16]
LIPS = [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
        78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308, 95, 88, 178, 87, 14, 317, 402, 318, 324]   # 40 lip points; [0]/[10] = mouth corners
N_PTS = 49 + len(LIPS)   # layout: 0-41 hands, 42-48 pose, 49-88 lips

train_csv = pd.read_csv(BASE / "train.csv")
with open(BASE / "sign_to_prediction_index_map.json", "r") as f:
    sign_map = json.load(f)

all_signs = sorted(sign_map)

if NUM_SIGNS >= len(all_signs):
    signs = all_signs
else:
    signs = sorted(np.random.RandomState(SIGN_SEED).choice(all_signs, NUM_SIGNS, replace=False).tolist())

sign2id = {s: i for i, s in enumerate(signs)}
df = train_csv[train_csv.sign.isin(signs)].reset_index(drop=True)
N_CLS = len(signs)

print(f"{N_CLS} signs, {len(df)} sequences, {df.participant_id.nunique()} signers")

assert len(df) > 0
first_path = BASE / df.path.iloc[0]
assert first_path.exists()
s0 = pd.read_parquet(first_path)
assert {"frame", "type", "landmark_index"}.issubset(s0.columns)
f0 = s0[s0.frame == s0.frame.iloc[0]]
assert len(f0) == 543
print("Layout OK")


In [ ]:
def load_one(path):
    d = pd.read_parquet(BASE / path, columns=["x", "y", "z"])
    n = len(d) // 543
    a = d[["x", "y", "z"]].to_numpy(np.float32)[: n * 543].reshape(n, 543, 3)
    return np.concatenate([a[:, LH0:LH0 + 21], a[:, RH0:RH0 + 21], a[:, POSE0:POSE0 + 33][:, POSE_KEEP], a[:, LIPS]], 1)

def normalize_body(kp):
    """Body frame: origin = mid-shoulder, unit = shoulder width."""
    ls, rs = kp[:, 43], kp[:, 44]
    mid = np.nanmean((ls + rs) / 2, axis=0)
    width = np.nanmean(np.linalg.norm((ls - rs)[:, :2], axis=-1))
    if not (np.isfinite(mid).all() and np.isfinite(width)) or width < 1e-3:
        mid = np.nanmean(kp[:, :42].reshape(-1, 3), axis=0)
        width = 0.25
        if not np.isfinite(mid).all():
            mid = np.zeros(3, np.float32)
    return (kp - mid) / width

def fill_hands(kpn, pres):
    """Linearly interpolate dropped hand detections between the first and last detected frame."""
    kpn, pres = kpn.copy(), pres.copy()
    for h in range(2):
        ok = np.where(pres[:, h])[0]
        if len(ok) < 2: continue
        t = np.arange(ok[0], ok[-1] + 1)
        for j in range(h * 21, h * 21 + 21):
            for d in range(3): kpn[t, j, d] = np.interp(t, ok, kpn[ok, j, d])
        pres[t, h] = True
    return kpn, pres

def process(path, T=T_CACHE):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kp = load_one(path)
        n = len(kp)
        if n == 0:
            return np.zeros((T, N_PTS, 3), np.float16), np.zeros((T, 2), bool)
        pres = (~np.isnan(kp[:, :42, 0])).reshape(n, 2, 21).any(-1)
        kpn = normalize_body(kp)
        kpn, pres = fill_hands(kpn, pres)
    act = np.where(pres.any(1))[0]                       # crop to the hand-present segment
    lo, hi = (max(act[0] - 2, 0), min(act[-1] + 2, n - 1)) if len(act) else (0, n - 1)
    idx = np.linspace(lo, hi, T).round().astype(int)
    c = np.clip(np.nan_to_num(kpn[idx], nan=0.0), -8, 8)
    return c.astype(np.float16), pres[idx]

cache = WORK / f"cache_{N_CLS}_{SIGN_SEED}_{T_CACHE}_lips.npz"
if cache.exists():
    z = np.load(cache); X, P = z["X"], z["P"]
else:
    t0 = time.time()
    out = Parallel(n_jobs=-1, batch_size=64)(delayed(process)(p) for p in tqdm(df.path, total=len(df)))
    X = np.stack([o[0] for o in out]); P = np.stack([o[1] for o in out])
    np.savez(cache, X=X, P=P)
    print(f"extracted in {time.time()-t0:.0f}s")

y_np = df.sign.map(sign2id).to_numpy()
print(X.shape, X.dtype, P.shape, f"hands present in {P.mean():.1%} of frames")

# left/right partner of every lip point, so mirroring keeps landmark identities consistent
from scipy.optimize import linear_sum_assignment
_lp = X[::10, :, 49:].astype(np.float32); _ok = np.abs(_lp).sum(-1) > 0
_mean = np.stack([(_lp[:, :, j] * _ok[:, :, j, None]).sum((0, 1)) / max(_ok[:, :, j].sum(), 1) for j in range(len(LIPS))])
_mir = _mean.copy(); _mir[:, 0] = -_mean[:, 0]
LIP_PERM = linear_sum_assignment(np.linalg.norm(_mean[None, :, :2] - _mir[:, None, :2], axis=-1))[1].tolist()
print(f"lips present in {_ok.mean():.1%} of frames | mean lip x offset {_mean[:, 0].mean():+.3f} (should be near 0)")

Xg = torch.from_numpy(X).to(DEVICE)
Pg = torch.from_numpy(P).to(DEVICE)
yg = torch.from_numpy(y_np).long().to(DEVICE)


## 2. Splits

In [ ]:
def random_split(seed, fr=(0.7, 0.15)):
    rs = np.random.RandomState(seed); y = df.sign.map(sign2id).to_numpy(); tr, va, te = [], [], []
    for c in range(N_CLS):
        i = rs.permutation(np.where(y == c)[0]); a, b = int(fr[0] * len(i)), int((fr[0] + fr[1]) * len(i))
        tr += i[:a].tolist(); va += i[a:b].tolist(); te += i[b:].tolist()
    t = lambda l: torch.tensor(l, dtype=torch.long, device=DEVICE)
    return t(tr), t(va), t(te)

def signer_split(seed, n_val=1, n_test=3):
    if SPLIT_MODE == "random": return random_split(seed)
    ids = np.array(sorted(df.participant_id.unique())); np.random.RandomState(seed).shuffle(ids)
    te, va, tr = ids[:n_test], ids[n_test:n_test + n_val], ids[n_test + n_val:]
    g = df.participant_id.to_numpy()
    t = lambda m: torch.from_numpy(np.where(m)[0]).long().to(DEVICE)
    return t(np.isin(g, tr)), t(np.isin(g, va)), t(np.isin(g, te))

def subsample_train(tr, seed, frac=1.0, n_signers=None):
    """Training subset that is identical for every arm given a seed. frac = class-stratified fraction, n_signers = keep only that many training signers."""
    rng = np.random.RandomState(1000 + seed); idx = tr.cpu().numpy()
    if n_signers is not None:
        g = df.participant_id.to_numpy()[idx]; keep = rng.permutation(np.unique(g))[:n_signers]; idx = idx[np.isin(g, keep)]
    if frac < 1.0:
        y = df.sign.map(sign2id).to_numpy()[idx]; sel = []
        for c in np.unique(y):
            ic = idx[y == c]; sel.append(rng.choice(ic, max(2, int(round(len(ic) * frac))), replace=False))
        idx = np.concatenate(sel)
    return torch.from_numpy(idx).long().to(DEVICE)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


## 3. Feature pipeline

In [ ]:
MIRROR_PERM = list(range(21, 42)) + list(range(0, 21)) + [42, 44, 43, 46, 45, 48, 47] + [49 + j for j in LIP_PERM]
def mirror(x, p):
    x = x[:, :, MIRROR_PERM].clone(); x[..., 0] = -x[..., 0]
    return x, p[:, :, [1, 0]]

def temporal_sample(x, p, T_out, frac=None):
    B, T = x.shape[:2]; dev = x.device
    ar = torch.arange(T_out, device=dev, dtype=torch.float32) / (T_out - 1)
    if frac is None:
        idx = ar.unsqueeze(0).expand(B, -1) * (T - 1)
    else:
        L = torch.empty(B, 1, device=dev).uniform_(*frac) * (T - 1)
        idx = torch.rand(B, 1, device=dev) * ((T - 1) - L) + L * ar
    i0 = idx.floor().long().clamp(0, T - 1); i1 = (i0 + 1).clamp(max=T - 1)
    w = (idx - i0.float()).view(B, T_out, 1, 1); bi = torch.arange(B, device=dev).unsqueeze(1)
    return x[bi, i0] * (1 - w) + x[bi, i1] * w, p[bi, idx.round().long().clamp(0, T - 1)]

def spatial_aug(x, p, rot_deg, scale, shift, jitter, mirror_p, shear=0.0):
    B, dev = x.shape[0], x.device
    if mirror_p > 0:
        xm, pm = mirror(x, p); f = torch.rand(B, device=dev) < mirror_p
        x = torch.where(f.view(B, 1, 1, 1), xm, x); p = torch.where(f.view(B, 1, 1), pm, p)
    a = torch.empty(B, device=dev).uniform_(-rot_deg, rot_deg) * math.pi / 180
    s = torch.empty(B, device=dev).uniform_(*scale)
    c, sn = (torch.cos(a) * s).view(B, 1, 1), (torch.sin(a) * s).view(B, 1, 1)
    x = torch.stack([x[..., 0] * c - x[..., 1] * sn, x[..., 0] * sn + x[..., 1] * c, x[..., 2] * s.view(B, 1, 1)], -1)
    if shear > 0:
        sh = torch.empty(B, 1, 1, device=dev).uniform_(-shear, shear)
        x = torch.stack([x[..., 0] + sh * x[..., 1], x[..., 1], x[..., 2]], -1)
    if shift > 0:  x = x + torch.empty(B, 1, 1, 3, device=dev).uniform_(-shift, shift)
    if jitter > 0:
        nz = torch.randn_like(x) * jitter; nz[:, :, 49:] *= 0.2; x = x + nz   # lips live at a small scale: less jitter
    return x, p

def time_mask(x, p, prob, max_frac=0.12):
    B, T = x.shape[:2]; dev = x.device
    ln = (torch.rand(B, 1, device=dev) * max_frac * T).long()
    st = (torch.rand(B, 1, device=dev) * (T - ln)).long()
    t = torch.arange(T, device=dev).unsqueeze(0)
    m = ((t >= st) & (t < st + ln) & (torch.rand(B, 1, device=dev) < prob))
    return x * (~m).view(B, T, 1, 1), p & (~m).unsqueeze(-1)

VEL = 10.0
TIPS, TRIU = [0, 4, 8, 12, 16, 20], torch.triu_indices(6, 6, 1)
HAND_D, POSE_D, LIP_D = 205, 42, 246
def hand_feats(h, ph):
    loc = h - h[:, :, :1]
    size = (h[:, :, 9] - h[:, :, 0]).norm(dim=-1).clamp(min=1e-2)
    loc = loc / size[..., None, None]
    v = torch.zeros_like(h); v[:, 1:] = (h[:, 1:] - h[:, :-1]) * VEL * (ph[:, 1:] * ph[:, :-1])[..., None, None]
    tp = h[:, :, TIPS]                                                   # wrist + 5 fingertips
    iu = TRIU.to(h.device)
    dd = (tp.unsqueeze(3) - tp.unsqueeze(2)).norm(dim=-1)[:, :, iu[0], iu[1]] / size[..., None]   # 15 pairwise distances
    return torch.cat([h.flatten(2), loc.flatten(2), v.flatten(2), ph.unsqueeze(-1), dd], -1)

def pose_feats(q):
    v = torch.zeros_like(q); v[:, 1:] = (q[:, 1:] - q[:, :-1]) * VEL
    return torch.cat([q.flatten(2), v.flatten(2)], -1)

def lip_feats(q, lp):
    """Lip shape (centred, mouth-width normalised) + its motion + mouth position/motion. lp = frame has lips."""
    c = q.mean(2, keepdim=True)
    size = (q[:, :, 0] - q[:, :, 10]).norm(dim=-1).clamp(min=1e-2)
    loc = (q - c) / size[..., None, None] * lp[..., None, None]
    both = (lp[:, 1:] * lp[:, :-1])[..., None, None]
    v = torch.zeros_like(loc); v[:, 1:] = (loc[:, 1:] - loc[:, :-1]) * VEL * both
    cv = torch.zeros_like(c); cv[:, 1:] = (c[:, 1:] - c[:, :-1]) * VEL * both
    return torch.cat([loc.flatten(2), v.flatten(2), c.flatten(2), cv.flatten(2)], -1)

def canon_dominant(x, p):
    """Mirror clips where the left hand is the more-used hand, so every signer looks right-handed."""
    flip = p[..., 0].float().sum(1) > p[..., 1].float().sum(1)
    xm, pm = mirror(x, p)
    return torch.where(flip.view(-1, 1, 1, 1), xm, x), torch.where(flip.view(-1, 1, 1), pm, p)

def make_inputs(cfg, xb, pb, train):
    x, p = xb.float(), pb
    x, p = temporal_sample(x, p, T_CACHE, cfg["warp"] if train else None)
    x, p = canon_dominant(x, p)
    lp = x[:, :, 49:].abs().sum((2, 3)) > 0                                   # frames where lips really exist
    if train:
        x, p = spatial_aug(x, p, **cfg["aug"]); x, p = time_mask(x, p, cfg["tmask"])
    lp = lp & (x[:, :, 49:].abs().sum((2, 3)) > 0)
    x = torch.cat([x[:, :, :42] * p.repeat_interleave(21, dim=2).unsqueeze(-1).to(x.dtype), x[:, :, 42:49],
                   x[:, :, 49:] * lp[..., None, None].to(x.dtype)], 2)
    pf = p.to(x.dtype)
    out = [hand_feats(x[:, :, :21], pf[..., 0]), hand_feats(x[:, :, 21:42], pf[..., 1])]
    if cfg.get("use_pose", False): out.append(pose_feats(x[:, :, 42:49]))
    if cfg.get("use_lips", False): out.append(lip_feats(x[:, :, 49:], lp.to(x.dtype)))
    return tuple(out)


## 4. Model

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, d, k=5, drop=0.1):
        super().__init__()
        self.norm, self.dw = nn.LayerNorm(d), nn.Conv1d(d, d, k, padding=k // 2, groups=d)
        self.pw, self.drop = nn.Linear(d, d), nn.Dropout(drop)
    def forward(self, x):
        y = self.dw(self.norm(x).transpose(1, 2)).transpose(1, 2)
        return x + self.drop(F.gelu(self.pw(y)))

class StreamEncoder(nn.Module):
    def __init__(self, d_in, d, drop):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_in, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(drop))
        self.conv = nn.Sequential(ConvBlock(d, 5, drop), ConvBlock(d, 5, drop))
        self.rnn, self.norm = nn.LSTM(d, d // 2, batch_first=True, bidirectional=True), nn.LayerNorm(d)
    def forward(self, x):
        h = self.conv(self.proj(x))
        return self.norm(h + self.rnn(h)[0])


class StatevectorCircuit(nn.Module):
    """Batched statevector simulation of a data-reuploading variational quantum circuit."""
    def __init__(self, n_qubits, q_layers):
        super().__init__()
        self.n, self.L = int(n_qubits), int(q_layers)
        self.weights = nn.Parameter(torch.rand(self.L, self.n, 3) * 2 * math.pi)
        N = 2 ** self.n
        f = torch.arange(N)
        for i in range(self.n):
            t = self.n - 1 - (i + 1) % self.n
            f = f ^ (((f >> (self.n - 1 - i)) & 1) << t)
        inv = torch.empty_like(f); inv[f] = torch.arange(N)
        self.register_buffer("perm", inv)

    def _rot(self):
        phi, th, om = self.weights.unbind(-1)
        c, s = torch.cos(th / 2), torch.sin(th / 2)
        e = lambda a: torch.polar(torch.ones_like(a), a)
        return torch.stack([torch.stack([e(-(phi + om) / 2) * c, -e((phi - om) / 2) * s], -1),
                            torch.stack([e(-(phi - om) / 2) * s,  e((phi + om) / 2) * c], -1)], -2)

    def forward(self, angles):
        B = angles.shape[0]
        a = angles.float().view(B, self.L, self.n)
        ca, sa = torch.cos(a / 2), torch.sin(a / 2)
        ry = torch.stack([torch.stack([ca, -sa], -1), torch.stack([sa, ca], -1)], -2).to(torch.complex64)
        U = self._rot().unsqueeze(0) @ ry
        sv = torch.zeros(B, 2 ** self.n, dtype=torch.complex64, device=angles.device)
        sv[:, 0] = 1
        for l in range(self.L):
            for w in range(self.n):
                sv = torch.einsum("bij,bljr->blir", U[:, l, w],
                                 sv.view(B, 2 ** w, 2, 2 ** (self.n - w - 1))).reshape(B, -1)
            sv = sv[:, self.perm]
        return sv.real.square() + sv.imag.square()


class ClassicalHead(nn.Module):
    """Classical stand-in for one circuit (control arm): angles -> probability-like vector, small MLP."""
    def __init__(self, n_in, n_out, hid=8):
        super().__init__(); self.net = nn.Sequential(nn.Linear(n_in, hid), nn.GELU(), nn.Linear(hid, n_out))
    def forward(self, a): return torch.softmax(self.net(a.float()), -1)


class QuantumFusionLayer(nn.Module):
    """Multi-head quantum fusion: the pooled feature is projected to angles for `heads` parallel VQCs,
    their measurement probabilities are concatenated, projected back and added as a gated residual."""
    def __init__(self, d_in, n_qubits=4, q_layers=2, heads=8, kind="quantum"):
        super().__init__()
        self.H, self.n_angles, self.n_outcomes = heads, n_qubits * q_layers, 2 ** n_qubits
        self.angle_proj = nn.Sequential(nn.Linear(d_in, 2 * heads * self.n_angles), nn.GELU(),
                                        nn.Linear(2 * heads * self.n_angles, heads * self.n_angles), nn.Tanh())
        self.circuits = nn.ModuleList([StatevectorCircuit(n_qubits, q_layers) if kind == "quantum" else ClassicalHead(self.n_angles, self.n_outcomes)
                                       for _ in range(heads)])
        self.post_proj = nn.Linear(heads * self.n_outcomes, d_in)
        self.norm = nn.LayerNorm(d_in)
        self.gate = nn.Parameter(torch.tensor(1.0))

    def forward(self, x):
        ang = (self.angle_proj(x) * math.pi).view(x.shape[0], self.H, self.n_angles)
        probs = torch.cat([c(ang[:, h].contiguous()) for h, c in enumerate(self.circuits)], -1)   # (B, H * 2^n)
        return self.norm(x + self.gate * self.post_proj(probs * self.n_outcomes - 1.0))


class HybridQuantumClassicalModel(nn.Module):
    """Hands (x2) + pose + lips stream encoders -> Transformer fusion -> attention pool -> quantum fusion -> head."""
    def __init__(self, n_cls, use_pose=False, use_lips=False, d=192, layers=3, heads=4, T=T_CACHE,
                 drop=0.15, n_qubits=4, q_layers=2, q_heads=8, use_quantum=True, circuit_kind="quantum"):
        super().__init__()
        self.use_quantum = use_quantum
        self.enc_l, self.enc_r = StreamEncoder(HAND_D, d, drop), StreamEncoder(HAND_D, d, drop)
        self.extra = nn.ModuleList(([StreamEncoder(POSE_D, d, drop)] if use_pose else []) +
                                   ([StreamEncoder(LIP_D, d, drop)] if use_lips else []))
        self.stream_emb = nn.Parameter(torch.randn(2 + len(self.extra), 1, d) * 0.02)
        self.time_emb = nn.Parameter(torch.randn(1, T, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, drop, activation="gelu", batch_first=True, norm_first=True)
        self.fusion = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.pool = nn.Linear(d, 1)
        self.quantum_fusion = QuantumFusionLayer(d, n_qubits, q_layers, q_heads, circuit_kind)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, n_cls))

    def forward(self, l, r, *more):
        s = [self.enc_l(l), self.enc_r(r)] + [e(t) for e, t in zip(self.extra, more)]
        z = self.fusion(torch.cat([t + self.time_emb + self.stream_emb[i] for i, t in enumerate(s)], 1))
        a = torch.softmax(self.pool(z).squeeze(-1), 1).unsqueeze(-1)
        pooled = (a * z).sum(1)
        return self.head(self.quantum_fusion(pooled) if self.use_quantum else pooled)


def build(cfg):
    m = HybridQuantumClassicalModel(N_CLS, cfg.get("use_pose", False), cfg.get("use_lips", False),
                                    d=cfg.get("d", 192), layers=cfg.get("layers", 3),
                                    n_qubits=cfg.get("n_qubits", 4), q_layers=cfg.get("q_layers", 2),
                                    q_heads=cfg.get("q_heads", 8), use_quantum=cfg.get("use_quantum", True),
                                    circuit_kind=cfg.get("circuit_kind", "quantum"))
    if cfg.get("freeze_circuit", False) and cfg.get("circuit_kind", "quantum") == "quantum":                 # control: circuits stay at random init, everything else trains
        for c in m.quantum_fusion.circuits:
            for q in c.parameters(): q.requires_grad_(False)
    return m

m = build({"use_pose": True, "use_lips": True})
n_params = sum(q.numel() for q in m.parameters())
n_quantum = sum(q.numel() for n, q in m.named_parameters() if "quantum" in n or "circuit" in n)
print(f"Total parameters: {n_params/1e6:.2f}M")
print(f"Quantum-related parameters: {n_quantum}")
print(f"Classical parameters: {(n_params - n_quantum)/1e6:.2f}M")


## 5. Training definitions

In [ ]:
OURS_AUG  = dict(rot_deg=15, scale=(0.80, 1.20), shift=0.1, jitter=0.005, mirror_p=0.0, shear=0.1)   # canon_dominant already makes everyone right-handed, so mirror aug only adds off-distribution clips

EXPERIMENTS = {
    "hybrid_quantum": dict(kind="ours", use_pose=False, warp=(0.7, 1.0), aug=OURS_AUG, tmask=0.3,
                           opt="adamw", lr=2e-3, wd=0.05, bs=128, epochs=EPOCHS, ls=0.1,
                           select="acc", patience=None, ema=0.995, tta=True,
                           n_qubits=4, q_layers=2, q_heads=8, use_quantum=True),
}

class EMA:
    def __init__(self, model, decay):
        self.decay, self.shadow = decay, copy.deepcopy(model).eval()
        for q in self.shadow.parameters(): q.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for e, m in zip(self.shadow.state_dict().values(), model.state_dict().values()):
            e.mul_(self.decay).add_(m.detach(), alpha=1 - self.decay) if e.dtype.is_floating_point else e.copy_(m)

@torch.no_grad()
def evaluate(model, idx, cfg, tta=False):
    model.eval(); lps = []
    for i in range(0, len(idx), 512):
        b = idx[i:i + 512]; xb, pb = Xg[b], Pg[b]
        with amp(): lp = F.log_softmax(model(*make_inputs(cfg, xb, pb, False)).float(), -1)
        if tta:
            xm, pm = mirror(xb.float(), pb)
            with amp(): lp2 = F.log_softmax(model(*make_inputs(cfg, xm, pm, False)).float(), -1)
            lp = torch.logsumexp(torch.stack([lp, lp2]), 0) - math.log(2)
        lps.append(lp)
    lp = torch.cat(lps); yt = yg[idx]; pred = lp.argmax(-1)
    return F.nll_loss(lp, yt).item(), (pred == yt).float().mean().item(), yt.cpu().numpy(), pred.cpu().numpy()


In [ ]:
# ============================================================
# FINAL RUN: Hybrid Quantum-Classical model
# Keeps the proven classical backbone + adds quantum fusion layer
# ============================================================
import matplotlib.pyplot as plt

USE_POSE = True
FINAL_CFG = dict(EXPERIMENTS["hybrid_quantum"], use_pose=USE_POSE, use_lips=True)
OUT_TAG = f"hybrid_quantum_{N_CLS}signs"


@torch.no_grad()
def predict_lp(model, idx, cfg):
    model.eval(); out = []
    for i in range(0, len(idx), 512):
        b = idx[i:i + 512]
        with amp(): out.append(F.log_softmax(model(*make_inputs(cfg, Xg[b], Pg[b], False)).float(), -1))
    return torch.cat(out)


def final_run(seed, cfg=FINAL_CFG, split_seed=None, tag=None, train_frac=1.0, n_signers=None, val_every=1, quiet=False, save=True):
    set_seed(seed); tr, va, te = signer_split(seed if split_seed is None else split_seed)
    tr = subsample_train(tr, seed, train_frac, n_signers)
    model = build(cfg).to(DEVICE); n_params = sum(q.numel() for q in model.parameters())
    steps = len(tr) // cfg["bs"]; assert steps > 0
    nd = lambda n, q: q.ndim < 2 or "circuit" in n or "emb" in n or n.endswith("gate")   # no weight decay on these
    opt = torch.optim.AdamW([
        {"params": [q for n, q in model.named_parameters() if not nd(n, q)], "weight_decay": cfg["wd"]},
        {"params": [q for n, q in model.named_parameters() if nd(n, q)], "weight_decay": 0.0}], lr=cfg["lr"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, cfg["lr"], total_steps=cfg["epochs"] * steps, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    ema = EMA(model, cfg["ema"]); evm = ema.shadow
    best, best_state, hist, t0 = None, None, [], time.time()
    for ep in range(cfg["epochs"]):
        model.train(); perm = tr[torch.randperm(len(tr), device=DEVICE)]; run_loss = 0.0
        for i in range(steps):
            b = perm[i * cfg["bs"]:(i + 1) * cfg["bs"]]
            with amp():
                logits = model(*make_inputs(cfg, Xg[b], Pg[b], True))
                loss = F.cross_entropy(logits.float(), yg[b], label_smoothing=cfg["ls"])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); ema.update(model)
            run_loss += loss.item() if i % 20 == 0 else 0
        if ep % val_every == 0 or ep == cfg["epochs"] - 1:
            vl, vacc, _, _ = evaluate(evm, va, cfg)
            hist.append(dict(epoch=ep, train_loss=run_loss / max(1, math.ceil(steps / 20)), val_loss=vl, val_acc=vacc))
        if ep == cfg["epochs"] - 1: best_state = {k: v.detach().clone() for k, v in evm.state_dict().items()}
    evm.load_state_dict(best_state)
    _, val_acc, _, _ = evaluate(evm, va, cfg)
    lp = predict_lp(evm, te, cfg); yt = yg[te]
    pred = lp.argmax(-1).cpu().numpy(); ytn = yt.cpu().numpy()
    top1 = float((pred == ytn).mean())
    pid = df.participant_id.to_numpy()[te.cpu().numpy()]
    per_signer = {int(p): float((pred[pid == p] == ytn[pid == p]).mean()) for p in np.unique(pid)}
    cls_acc = np.array([(pred[ytn == c] == c).mean() if (ytn == c).any() else np.nan for c in range(N_CLS)])
    if save: torch.save(best_state, WORK / f"{tag or OUT_TAG}_seed{seed}.pt")
    r = dict(seed=seed, n_train=len(tr), params=n_params, val_acc=val_acc, test_acc=top1, secs=time.time() - t0,
             per_signer=per_signer, cls_acc=cls_acc, history=hist)
    psg = ['%.2f' % v for v in per_signer.values()]
    if not quiet: print(f"[seed {seed}] test accuracy {top1*100:.1f}% | per-signer {psg} | {r['secs']:.0f}s")
    return r


## 6. Train the deployment model and export

In [ ]:
# ============================================================
# DEPLOYMENT TRAINING + EXPORT
# ============================================================
_orig_signer_split = signer_split
signer_split = lambda seed, n_val=1, n_test=1: _orig_signer_split(seed, n_val, n_test)     # 19 train / 1 val / 1 sanity-test signers
tag = f"deploy_{N_CLS}signs"
r = final_run(0, tag=tag, split_seed=0)                                                    # saves the EMA weights to {tag}_seed0.pt
signer_split = _orig_signer_split

state = torch.load(WORK / f"{tag}_seed0.pt", map_location="cpu")
bundle = dict(
    state=state, signs=list(signs), lip_perm=[int(i) for i in LIP_PERM], T=T_CACHE,
    model_cfg=dict(use_pose=True, use_lips=True, d=192, layers=3, n_qubits=4, q_layers=2, q_heads=8, use_quantum=True),
    info=dict(n_signs=N_CLS, n_train=int(r["n_train"]), params=int(r["params"]), val_acc=float(r["val_acc"]),
              sanity_test_acc=float(r["test_acc"]), epochs=int(EPOCHS)))
out_path = WORK / f"slt_deploy_{N_CLS}signs.pt"
torch.save(bundle, out_path)
print(f"\nsaved {out_path}  ({out_path.stat().st_size/1e6:.1f} MB)")
print(f"validation signer accuracy {r['val_acc']*100:.1f}% | sanity-test signer accuracy {r['test_acc']*100:.1f}%  (one signer each, so noisy)")
print("vocabulary:", ", ".join(signs))
print("Download the .pt file from the Output tab, then follow DEPLOY.md.")